# 04 - Treinamento e Avaliação do Modelo

Neste notebook, treinamos e comparamos diferentes modelos de regressão para estimar a pegada de carbono. Começamos com um baseline linear e progredimos para modelos de conjunto (*ensemble*) como Random Forest e Gradient Boosting.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import os

# Semente aleatória para reprodutibilidade
np.random.seed(42)

## 1. Carregamento e Preparação dos Dados

Carregamos o dataset e preparamos as features temporais baseadas nos insights do EDA.

In [2]:
# Carregar o dataset
data_path = 'https://raw.githubusercontent.com/carbon-footprint-analysis/carbon-footprint-analysis/main/data/processed/synthetic_energy_emissions_dataset.csv'
df = pd.read_csv(data_path)

# Engenharia de features temporal
df['data'] = pd.to_datetime(df['data'])
df['mes'] = df['data'].dt.month

def get_season(month):
    if mes in [12, 1, 2]: return 'Verao'
    if mes in [3, 4, 5]: return 'Outono'
    if mes in [6, 7, 8]: return 'Inverno'
    return 'Primavera'

df['season'] = df['month'].apply(get_season)

# Seleção de atributos e target
target = 'co2_emission'
features = ['energy_kwh', 'state', 'usage_type', 'energy_source', 'month', 'season']

X = df[features]
y = df[target]

# Divisão Treino/Teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Treino: {X_train.shape}, Teste: {X_test.shape}")


KeyError: 'month'

## 2. Pipeline de Pré-processamento

Definimos o processamento comum para todos os modelos.

In [ ]:
# Definição das colunas
numeric_features = ['energy_kwh', 'month']
categorical_features = ['state', 'usage_type', 'energy_source', 'season']

# Transformers
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)


## 3. Modelo Baseline: Regressão Linear

Treinamos o modelo simples para referência.

In [ ]:
# Pipeline Linear
model_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

model_lr.fit(X_train, y_train)
y_pred_lr = model_lr.predict(X_test)

print(f"Regressão Linear - R²: {r2_score(y_test, y_pred_lr):.4f}")


## 4. Modelagem Avançada

Exploramos modelos de ensemble para capturar relações complexas.

### 4.1. Random Forest Regressor

In [ ]:
# Pipeline Random Forest
model_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1))
])

model_rf.fit(X_train, y_train)
y_pred_rf = model_rf.predict(X_test)

print(f"Random Forest - R²: {r2_score(y_test, y_pred_rf):.4f}")


### 4.2. Gradient Boosting Regressor

In [ ]:
# Pipeline Gradient Boosting
model_gb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42))
])

model_gb.fit(X_train, y_train)
y_pred_gb = model_gb.predict(X_test)

print(f"Gradient Boosting - R²: {r2_score(y_test, y_pred_gb):.4f}")


## 5. Comparação de Performance e Seleção do Melhor Modelo

Analisamos as métricas para decidir qual modelo utilizar em produção.

In [ ]:
def get_metrics(y_true, y_pred, name):
    return {
        'Model': name,
        'R2': r2_score(y_true, y_pred),
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred))
    }

results = pd.DataFrame([
    get_metrics(y_test, y_pred_lr, 'Linear Regression'),
    get_metrics(y_test, y_pred_rf, 'Random Forest'),
    get_metrics(y_test, y_pred_gb, 'Gradient Boosting')
])

display(results.sort_values('R2', ascending=False))

# Identificação automática do melhor modelo
best_row = results.sort_values('R2', ascending=False).iloc[0]
best_model_name = best_row['Model']
print(f"\n🏆 O melhor modelo baseado no R² é: {best_model_name}")

# Atribuição do pipeline campeão
if best_model_name == 'Gradient Boosting':
    best_pipeline = model_gb
elif best_model_name == 'Random Forest':
    best_pipeline = model_rf
else:
    best_pipeline = model_lr

# Plot comparativo
plt.figure(figsize=(10, 5))
sns.barplot(data=results, x='Model', y='R2', palette='viridis')
plt.axhline(0.9, color='red', linestyle='--', label='Meta de Performance')
plt.title('Comparação de Performance (R²)')
plt.ylim(0, 1.1)
plt.legend()
plt.show()


### Conclusão Intermediária

O melhor modelo é aquele que apresenta o maior **R²** (capacidade de explicação da variância) e o menor **MAE/RMSE** (erro médio). Geralmente, o **Gradient Boosting** ou o **Random Forest** superam significativamente a Regressão Linear neste caso.

## 6. Importância das Features (Melhor Modelo)

Visualizamos quais variáveis mais influenciam o modelo campeão.

In [ ]:
# Extração dos nomes das colunas após OneHotEncoding
ohe_feature_names = best_pipeline.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(categorical_features)
feature_names = numeric_features + list(ohe_feature_names)

# Importâncias
importances = best_pipeline.named_steps['regressor'].feature_importances_
feature_importance_df = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=feature_importance_df.head(10), x='importance', y='feature', palette='magma')
plt.title(f'Top 10 Variáveis mais Importantes ({best_model_name})')
plt.show()


## 7. Teste de Robustez (Stress Test)

Validamos o modelo campeão contra pequenas variações/ruídos nos dados de entrada.

In [ ]:
# Simulação de ruído (5% de desvio padrão do consumo)
X_test_noisy = X_test.copy()
noise_scale = X_test_noisy['energy_kwh'] * 0.05
noise = np.random.normal(0, 1, size=len(X_test_noisy)) * noise_scale
X_test_noisy['energy_kwh'] += noise

# Predições no melhor modelo
y_pred_noisy = best_pipeline.predict(X_test_noisy)

# Métricas ruidosas
r2_orig = r2_score(y_test, best_pipeline.predict(X_test))
r2_noisy = r2_score(y_test, y_pred_noisy)
mae_noisy = mean_absolute_error(y_test, y_pred_noisy)

print(f"--- Teste de Robustez no {best_model_name} (5% Ruído) ---")
print(f"R² Original (Sem Ruído): {r2_orig:.4f}")
print(f"R² Ruidoso (Com 5% de Ruído): {r2_noisy:.4f}")
print(f"Queda Relativa no R²: {((r2_orig - r2_noisy) / r2_orig) * 100:.2f}%")
print(f"Novo MAE: {mae_noisy:.2f} kg CO2")


## 8. Persistência do Melhor Modelo para Deployment

Salvamos o modelo campeão para ser utilizado pelo script de predição.

In [ ]:
# Definir o nome do arquivo final
model_export_path = os.path.join('..', 'models', '../models/best_carbon_footprint_model.joblib')

# Garantir que a pasta models exista
os.makedirs(os.path.dirname(model_export_path), exist_ok=True)

# Salvar o pipeline completo (preprocessor + regressor)
joblib.dump(best_pipeline, model_export_path)

print(f"✅ Modelo '{best_model_name}' exportado com sucesso para: {model_export_path}")
